In [2]:
import os

In [3]:
%pwd

'c:\\Users\\SAAD TARIQ\\github_repositories\\wine-quality\\notebooks'

In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\SAAD TARIQ\\github_repositories\\wine-quality'

In [12]:
import os
import joblib
import pandas as pd
from dataclasses import dataclass
from pathlib import Path
from sklearn.linear_model import ElasticNet
from src.wine_quality_prediction.constants import *
from src.wine_quality_prediction.utils.common import read_yaml, create_directories
from src.wine_quality_prediction import logger

In [9]:
@dataclass
class ModelTrainerConfig:
    root_directory: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str

In [15]:
class ConfigurationManager:
    def __init__(self, config_file_path=CONFIG_FILE_PATH,
                 params_file_path=PARAMS_FILE_PATH,
                 schema_file_path=SCHEMA_FILE_PATH):
        self.config_file_path = read_yaml(path_to_yaml=config_file_path)
        self.params_file_path = read_yaml(path_to_yaml=params_file_path)
        self.schema_file_path = read_yaml(path_to_yaml=schema_file_path)

        create_directories([self.config_file_path.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config_file_path.model_trainer
        params = self.params_file_path.elastic_net
        schema = self.schema_file_path.target_column

        create_directories([config.root_directory])

        model_trainer_config = ModelTrainerConfig(
            root_directory=config.root_directory,
            train_data_path=config.train_data_path,
            test_data_path=config.test_data_path,
            model_name=config.model_name,
            alpha=params.alpha,
            l1_ratio=params.l1_ratio,
            target_column=schema.name,
        )

        return model_trainer_config

In [16]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)

        train_x = train_data.drop([self.config.target_column], axis=1)
        test_x = test_data.drop([self.config.target_column], axis=1)
        train_y = train_data[[self.config.target_column]]
        test_y = test_data[[self.config.target_column]]

        model = ElasticNet(
            alpha=self.config.alpha,
            l1_ratio=self.config.l1_ratio,
            random_state=42
        )
        model.fit(train_x, train_y)

        joblib.dump(
            model, os.path.join(
                self.config.root_directory, self.config.model_name
            )
        )

In [17]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()
except Exception as e:
    raise e

[2026-03-05 03:16:03,661] INFO: common: YAML file 'config\config.yaml' read successfully.]
[2026-03-05 03:16:03,662] INFO: common: YAML file 'params.yaml' read successfully.]
[2026-03-05 03:16:03,663] INFO: common: YAML file 'schema.yaml' read successfully.]
[2026-03-05 03:16:03,664] INFO: common: Directory 'artifacts' created successfully or already exists.]
[2026-03-05 03:16:03,665] INFO: common: Directory 'artifacts/model_trainer' created successfully or already exists.]
